# FastAPI API Overview

A runnable walkthrough of the core `FastAPI` building blocks:
- Path operations
- Request validation
- Dependency injection
- Error handling
- The automatic interactive docs

`FastAPI` apps are served by `uvicorn`
- This notebook uses `fastapi.testclient.TestClient` instead, which drives the app in-process without opening a real socket, so every cell runs instantly.

**What you will learn:**
- How path and query parameters are typed and validated
- How request bodies are validated with `pydantic` models
- How to share logic across endpoints with `Depends()`
- How to return structured errors with `HTTPException`
- Where the automatic `/docs`, `/redoc`, and `/openapi.json` come from

Related notebook: `fastapi.example.ipynb` runs a complete app with a real
`uvicorn` server and real HTTP calls over the network.

In [1]:
!pip install --quiet -r tutorial_requirements.txt

In [2]:
%load_ext autoreload
%autoreload 2

import logging

import fastapi
import fastapi.testclient

import fastapi_utils
import helpers.hdbg as hdbg

hdbg.init_logger(verbosity=logging.INFO)
_LOG = logging.getLogger(__name__)

INFO  > /usr/local/lib/python3.12/site-packages/ipykernel_launcher.py -f /root/.local/share/jupyter/runtime/kernel-ffa4f592-5fc7-4c2c-89a5-db89b1907971.json


/usr/local/lib/python3.12/site-packages/fastapi/testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


## 1. Path Operations

- A path operation is a Python function decorated with an HTTP method and a path, e.g. `@app.get("/items/{item_id}")`
- `FastAPI` reads the function's type hints to know how to parse and validate each argument

In [13]:
# Create a FastAPI application.
demo_app = fastapi.FastAPI()

# Create a client that connects to the API directly, without going through sockets and HTTP (useful for unit testing).
demo_client = fastapi.testclient.TestClient(demo_app) 

# - A path operation is an "endpoint":
#   - One URL path: `/`
#   - One HTTP method: `get`
#   - The function that handles it: `read_root()`
@demo_app.get("/")
def read_root() -> dict:
    """
    Return a simple greeting.
    """
    return {"message": "Hello, World"}

In [11]:
# Call the API.
response = demo_client.get("/")
_LOG.info("GET / -> %s %s", response.status_code, response.json())

INFO  GET / -> 200 {'message': 'Hello, World'}


In [14]:
# An API "items" accepting an integer id.
@demo_app.get("/items/{item_id}")
def read_item(item_id: int) -> dict:
    """
    Echo back a path parameter, coerced to `int`.
    """
    return {"item_id": item_id}

In [15]:
# Call the API.
path = "/items/42"
response = demo_client.get(path)
_LOG.info("GET %s -> %s %s", path, response.status_code, response.json())

INFO  GET /items/42 -> 200 {'item_id': 42}


In [8]:
# A non-integer path segment fails validation before `read_item()` ever runs.
response = demo_client.get("/items/not_a_number")
_LOG.info("GET /items/not_a_number -> %s", response.status_code)
_LOG.info("Validation error detail: %s", response.json()["detail"][0]["msg"])

INFO  GET /items/not_a_number -> 422
INFO  Validation error detail: Input should be a valid integer, unable to parse string as an integer


## 2. Query Parameters

Function arguments that are not part of the path become query parameters.
A default value makes the parameter optional.

In [17]:
@demo_app.get("/items/")
def list_items(skip: int = 0, limit: int = 10) -> dict:
    """
    Report the pagination parameters that were parsed.
    """
    return {"skip": skip, "limit": limit}

In [18]:
# Explicit `skip` and `limit` values are parsed from the query string.
response = demo_client.get("/items/", params={"skip": 5, "limit": 20})
_LOG.info("GET /items/?skip=5&limit=20 -> %s", response.json())

INFO  GET /items/?skip=5&limit=20 -> {'skip': 5, 'limit': 20}


In [19]:
# Omitting both parameters falls back to the function's default values.
response = demo_client.get("/items/")
_LOG.info("GET /items/ (defaults) -> %s", response.json())

INFO  GET /items/ (defaults) -> {'skip': 0, 'limit': 10}


## 3. Request Body and Validation

// TODO(ai_gp): Do not use utils but create all the needed code here

- A request body is described as a `pydantic` model
- `fastapi_utils` defines `BookCreate` and `Book` for the tutorial
    - Reusing them here keeps this notebook and `fastapi.example.ipynb` consistent

In [ ]:
@demo_app.post("/books", response_model=fastapi_utils.Book, status_code=201)
def create_book(payload: fastapi_utils.BookCreate) -> fastapi_utils.Book:
    """
    Echo the validated payload back as a `Book` with a fixed ID.

    A real implementation would persist the book; see
    `fastapi_utils.create_book_app()` for that version.
    """
    return fastapi_utils.Book(id=1, **payload.model_dump())


response = demo_client.post(
    "/books", json={"title": "Fluent Python", "author": "Luciano Ramalho", "year": 2015}
)
_LOG.info("POST /books (valid) -> %s %s", response.status_code, response.json())

In [ ]:
# Omitting a required field fails validation before `create_book()` runs.
response = demo_client.post("/books", json={"title": "Missing Fields"})
_LOG.info("POST /books (invalid) -> %s", response.status_code)
for error in response.json()["detail"]:
    _LOG.info("  %s: %s", error["loc"], error["msg"])

## 4. Dependency Injection

- `Depends()` lets multiple endpoints share the same parameter-parsing logic instead of repeating it
- `FastAPI` calls the dependency function first and passes its return value into the endpoint

In [ ]:
def pagination_params(
    skip: int = fastapi.Query(0, ge=0),
    limit: int = fastapi.Query(10, ge=1, le=100),
) -> dict:
    """
    Parse and validate pagination parameters shared across endpoints.
    """
    return {"skip": skip, "limit": limit}


@demo_app.get("/books")
def list_books_demo(
    pagination: dict = fastapi.Depends(pagination_params),
) -> dict:
    """
    Show the pagination values resolved by the shared dependency.
    """
    return pagination


response = demo_client.get("/books", params={"limit": 5})
_LOG.info("GET /books?limit=5 -> %s", response.json())

# An out-of-range value is rejected by the dependency's own `Query()` bounds.
response = demo_client.get("/books", params={"limit": 500})
_LOG.info("GET /books?limit=500 -> %s", response.status_code)

## 5. Error Handling with HTTPException

`fastapi_utils.create_book_app()` builds a small catalog API on top of the
same models. Looking up a missing book raises `HTTPException`, which
`FastAPI` turns into a JSON error response with the given status code.

In [ ]:
catalog_app = fastapi_utils.create_book_app()
catalog_client = fastapi.testclient.TestClient(catalog_app)

response = catalog_client.get("/books/999")
_LOG.info("GET /books/999 -> %s %s", response.status_code, response.json())

## 6. Automatic Interactive Docs

Every `FastAPI` app serves three documentation endpoints for free, derived
from the same type hints used for validation:
- `/docs`: interactive Swagger UI
- `/redoc`: read-only ReDoc reference
- `/openapi.json`: the raw OpenAPI schema

`fastapi.example.ipynb` opens these in a browser against a real server;
here, `TestClient` fetches the schema directly.

In [ ]:
response = catalog_client.get("/openapi.json")
schema = response.json()
_LOG.info("OpenAPI title: %s", schema["info"]["title"])
_LOG.info("Registered paths: %s", sorted(schema["paths"].keys()))

## 7. Testing with TestClient

`TestClient` is not just a notebook convenience: it is the same tool used
in automated `pytest` tests, since it drives the app in-process.

In [ ]:
def test_health_check() -> None:
    """
    Confirm the catalog app reports itself as healthy.
    """
    result = catalog_client.get("/health")
    assert result.status_code == 200
    assert result.json() == {"status": "ok"}


test_health_check()
_LOG.info("test_health_check() passed.")